# Flash Attention — Dao et al. 2022

**Paper:** FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness

## The Problem with Standard Attention

Standard attention computes `softmax(QK^T / sqrt(d_k)) @ V`.
The issue: `QK^T` produces an `[N x N]` matrix where N = sequence length.

- N=1024  → 1M elements → fine
- N=8192  → 67M elements → ~256MB just for one layer
- N=32768 → 1B elements → out of memory

Also, writing this giant matrix to GPU HBM (high-bandwidth memory) and reading it back is the bottleneck — not the FLOPs.

## The Flash Attention Idea

**Tiling:** split Q, K, V into blocks and compute attention block-by-block, keeping running statistics in fast SRAM instead of writing the full N×N matrix to HBM.

**Key insight:** softmax can be computed incrementally using the log-sum-exp trick.
You never materialize the full N×N attention matrix.

```
Standard attention:
  HBM read:  Q, K, V
  SRAM:      compute QK^T block        → write full N×N to HBM
  HBM read:  N×N matrix
  SRAM:      softmax + multiply V      → write output to HBM
  Memory:    O(N²)  ← the problem

Flash Attention:
  HBM read:  Q block, K block, V block  (tiled)
  SRAM:      compute partial attention, accumulate output
             never write N×N matrix to HBM
  Memory:    O(N)   ← the fix
```

## Resources

| Resource | Link |
|---|---|
| Original paper (Dao et al. 2022) | https://arxiv.org/abs/2205.14135 |
| FlashAttention-2 paper (Dao 2023) | https://arxiv.org/abs/2307.08691 |
| FlashAttention-3 (H100 optimized) | https://tridao.me/blog/2024/flash3/ |
| Official GitHub (flash-attn) | https://github.com/Dao-AILab/flash-attention |
| Triton fused attention tutorial | https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html |
| ELI5 Medium walkthrough | https://gordicaleksa.medium.com/eli5-flash-attention-5c44017022ad |

In [ ]:
import torch
import torch.nn.functional as F
import math

# --- Standard attention for reference ---
def standard_attention(Q, K, V, mask=None):
    """
    Q, K, V: [batch, heads, seq_len, d_k]
    Materializes the full N×N attention matrix — memory O(N²)
    """
    d_k    = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # [b, h, N, N] ← this is the problem
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V


batch, heads, seq_len, d_k = 1, 4, 64, 32
Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

out_standard = standard_attention(Q, K, V)
print('Standard attention output shape:', out_standard.shape)
print(f'N×N matrix size: {seq_len}×{seq_len} = {seq_len*seq_len:,} elements')

## The Log-Sum-Exp (LSE) Trick — enables tiling

Softmax requires knowing the sum of ALL exp(scores) before normalizing.
With tiling, we process blocks and don't see all scores at once.

The trick: maintain a running maximum `m` and running sum `l`.
When we see a new block, we can *update* these statistics without recomputing from scratch.

```
For block 1: m1 = max(scores1),  l1 = sum(exp(scores1 - m1))
For block 2: m2 = max(scores2)
             m_new = max(m1, m2)
             l_new = exp(m1 - m_new) * l1 + exp(m2 - m_new) * sum(exp(scores2 - m2))
```

The output accumulator is updated the same way — weighted combination of partial outputs.

In [ ]:
def flash_attention_v1(Q, K, V, block_size=16):
    """
    Pure PyTorch Flash Attention — conceptual implementation.
    Matches the paper algorithm. Does NOT achieve the real CUDA speedup
    (that requires custom CUDA/Triton kernels), but demonstrates the algorithm.

    Q, K, V: [seq_len, d_k]  (single head, no batch for clarity)
    """
    seq_len, d_k = Q.shape
    scale        = 1.0 / math.sqrt(d_k)

    # Output accumulator and softmax statistics
    O = torch.zeros(seq_len, d_k)           # output
    l = torch.zeros(seq_len)                # running sum of exp (denominator)
    m = torch.full((seq_len,), float('-inf'))  # running max

    # Iterate over K, V blocks (outer loop)
    for j in range(0, seq_len, block_size):
        K_block = K[j:j+block_size]   # [block_size, d_k]
        V_block = V[j:j+block_size]   # [block_size, d_k]

        # Iterate over Q blocks (inner loop)
        for i in range(0, seq_len, block_size):
            Q_block = Q[i:i+block_size]   # [block_size, d_k]

            # Compute attention scores for this block pair
            S_block = Q_block @ K_block.T * scale   # [block_q, block_k]

            # Block max and sum for numerical stability
            m_block = S_block.max(dim=-1).values    # [block_q]
            P_block = torch.exp(S_block - m_block.unsqueeze(-1))  # [block_q, block_k]
            l_block = P_block.sum(dim=-1)           # [block_q]

            # Update running statistics using the online softmax trick
            m_new = torch.maximum(m[i:i+block_size], m_block)
            l_new = (torch.exp(m[i:i+block_size] - m_new) * l[i:i+block_size]
                     + torch.exp(m_block - m_new) * l_block)

            # Update output accumulator
            O[i:i+block_size] = (
                (torch.exp(m[i:i+block_size] - m_new) * l[i:i+block_size]).unsqueeze(-1) * O[i:i+block_size]
                + torch.exp(m_block - m_new).unsqueeze(-1) * (P_block @ V_block)
            ) / l_new.unsqueeze(-1)

            # Save updated statistics
            m[i:i+block_size] = m_new
            l[i:i+block_size] = l_new

    return O


# Test: flash attention output should match standard attention
seq, d_k = 64, 32
q = torch.randn(seq, d_k)
k = torch.randn(seq, d_k)
v = torch.randn(seq, d_k)

out_flash    = flash_attention_v1(q, k, v, block_size=16)
out_standard = F.softmax(q @ k.T / math.sqrt(d_k), dim=-1) @ v

print('Flash matches standard:', torch.allclose(out_flash, out_standard, atol=1e-5))
print('Max diff:', (out_flash - out_standard).abs().max().item())

## Memory comparison

In [ ]:
# Memory usage comparison at different sequence lengths
for N in [512, 1024, 4096, 8192]:
    d_k    = 64
    # Standard attention: stores NxN float32 matrix
    std_mb = N * N * 4 / 1e6
    # Flash attention: only stores blocks in SRAM, output O(N)
    flash_mb = N * d_k * 4 / 1e6
    print(f'N={N:5d} | Standard: {std_mb:8.1f} MB | Flash: {flash_mb:5.1f} MB | Ratio: {std_mb/flash_mb:.0f}x')

## Using the real Flash Attention library on Colab GPU

```bash
# On Colab with GPU:
pip install flash-attn --no-build-isolation
```

```python
from flash_attn import flash_attn_func

# Input shape: [batch, seq_len, num_heads, head_dim]
Q = torch.randn(2, 1024, 8, 64, dtype=torch.float16, device='cuda')
K = torch.randn(2, 1024, 8, 64, dtype=torch.float16, device='cuda')
V = torch.randn(2, 1024, 8, 64, dtype=torch.float16, device='cuda')

out = flash_attn_func(Q, K, V, causal=True)   # causal=True for GPT-style
print(out.shape)   # [2, 1024, 8, 64]
```

Note: flash-attn requires **CUDA + float16/bfloat16**. Does NOT work on CPU or float32.
On Colab: Runtime → T4 GPU → use float16.